# TP2 · 03 · Pacote da submissão (Internet **On**, sem GPU)

O notebook de submissão (04) roda **sem internet**, exigência da competição. Este notebook prepara
o que ele precisa e grava tudo na sua saída:
* `code/`: cópia do código do repositório (src, scripts, configs, splits);
* `code/priors.json`: prevalências e taxa de subamostragem calculadas do `train.csv` (só a partição
  de treino). A submissão não pode usar o treino da competição, então leva esses números prontos;
* `wheels/`: decodificadores de DICOM comprimido (pylibjpeg, GDCM), para instalar sem internet.

**Input:** a competição **RSNA STR Pulmonary Embolism Detection** (só o `train.csv` é lido).
**Settings:** Accelerator **None**, Internet **On**. Rode com **Save & Run All**.


In [ ]:
# ===== TP2 · pacote da submissão =====
import os, shutil, subprocess, sys
W = "/kaggle/working"
REPO = f"{W}/_repo"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", "claude/peaceful-curie-mqrljz",
                "https://github.com/Silvarawenne/ap2-topicos.git", REPO], check=True)
CODE = f"{W}/code"
shutil.rmtree(CODE, ignore_errors=True)
for d in ("src", "scripts", "configs", "splits"):
    shutil.copytree(f"{REPO}/{d}", f"{CODE}/{d}")
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(commit, file=open(f"{CODE}/COMMIT.txt", "w"))
shutil.rmtree(REPO)

# prevalências do treino -> code/priors.json (lidas pelo 13_kaggle_infer.py na submissão)
import glob, json
tc = None
for prof in range(0, 3):
    h = sorted(glob.glob("/kaggle/input/" + "*/" * prof + "rsna-str-pulmonary-embolism-detection*/train.csv"))
    if h:
        tc = h[0]
        break
if tc is None:
    raise SystemExit("adicione a competição como Input (train.csv não encontrado)")
sys.path.insert(0, f"{CODE}/src")
import yaml, pandas as pd
from pe_tp2 import submission as S
cfg = yaml.safe_load(open(f"{CODE}/configs/config.yaml"))
priors = S.train_priors(pd.read_csv(tc), pd.read_csv(f"{CODE}/{cfg['split']['file']}"), cfg["tp2"]["neg_ratio"])
json.dump(priors, open(f"{CODE}/priors.json", "w"), indent=2)
print("priors:", json.dumps(priors))

PKGS = ["pydicom>=3.0", "pylibjpeg>=2.0", "pylibjpeg-libjpeg>=2.1", "pylibjpeg-openjpeg>=2.3", "python-gdcm>=3.0.24"]
WH = f"{W}/wheels"
shutil.rmtree(WH, ignore_errors=True)
subprocess.run([sys.executable, "-m", "pip", "download", "-q", "-d", WH, "--only-binary=:all:", *PKGS], check=True)
# conferência: instala só do pacote, sem índice (como o notebook 04 fará)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH, *PKGS], check=True)
print("commit do código:", commit)
print("wheels:", sorted(os.listdir(WH)))
